In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer 

In [4]:
model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [5]:
dataset = load_dataset(
    "text",
    data_files="data/train.txt",
    split="train"
)


In [6]:
def tokenize_function(example):
    return tokenizer(example["text"])

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

In [7]:
print(tokenized_dataset[0])

{'input_ids': [46208, 4288, 374, 458, 15235, 28383, 448, 264, 3746, 75904, 8841, 4752, 5670, 40227, 24514, 5942, 4751, 1091, 4936, 53852, 6350, 15235, 33356, 13, 220], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [8]:
block_size = 32


def group_texts(examples):

    concatenated_examples = {}

    for key in examples.keys():
        concatenated_examples[key] = sum(
            examples[key],
            []
        )

    total_length = len(
        concatenated_examples["input_ids"]
    )

    total_length = (
        total_length // block_size
    ) * block_size

    result = {}

    for key, tokens in concatenated_examples.items():

        result[key] = [
            tokens[i:i + block_size]
            for i in range(
                0,
                total_length,
                block_size
            )
        ]

    result["labels"] = result["input_ids"].copy()

    return result

In [9]:
lm_dataset = tokenized_dataset.map(
    group_texts,
    batched=True
)

In [7]:
training_args = TrainingArguments(
    output_dir = "./outputs",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate = 5e-5,
    logging_steps = 1,
    save_strategy= "epoch",
    report_to = "none"
 )

In [11]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = lm_dataset
)

In [12]:
trainer.train()

c:\Users\Mahesh.shriram\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,4.110615
2,4.381512
3,4.309369
4,3.859018
5,2.890084
6,1.446435
7,1.541411
8,1.349382
9,1.326453
10,1.197940


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\Mahesh.shriram\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\Mahesh.shriram\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=15, training_loss=1.9842503269513447, metrics={'train_runtime': 10832.2243, 'train_samples_per_second': 0.003, 'train_steps_per_second': 0.001, 'total_flos': 2061493125120.0, 'train_loss': 1.9842503269513447, 'epoch': 3.0})

In [13]:
trainer.save_model("./outputs/final_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
tokenizer.save_pretrained("./outputs/final_model")

('./outputs/final_model\\tokenizer_config.json',
 './outputs/final_model\\chat_template.jinja',
 './outputs/final_model\\tokenizer.json')